# Snapget — tự động cào ảnh + tự động gán nhãn cho tập VN-train (Google Colab)

Chạy `ml/scripts/17_crawl_autolabel.py` theo **`ml/DATASET_SPEC.md`**: tìm ảnh theo từ khoá bối cảnh Việt Nam (`ml/crawl_queries.yaml`) trên **Openverse + Wikimedia Commons** (ảnh Creative Commons, có license) và **DuckDuckGo** (tìm tiếng Việt) → lọc chất lượng (cạnh dài ≥ 1024, bỏ nền trắng studio, bỏ trang stock/TMĐT hay có watermark) → khử trùng → **gán box bằng 2 model** (YOLO11x COCO + YOLO-World với prompt "ghế nhựa", "ly trà sữa", "xe ga"… và hard negative "đồng hồ đeo tay", "nón lá"…) → ảnh chắc chắn vào `train/`, ảnh nghi vấn vào `review/` để người sửa trên Roboflow.

**Runtime → Change runtime type → T4 GPU.** Kết quả lưu thẳng lên Drive (`WORK/data/vn_web/`) → Colab ngắt thì **chạy lại ô 3, script tự chạy tiếp** (bỏ URL đã xử lý, đếm ảnh đã có).

> ⚠️ Ảnh Internet **chỉ vào VN-train**, không bao giờ vào test. Ảnh `accept` vào thẳng train không qua người duyệt từng ảnh → **xem mẫu ~10%** ở ô 4 (DATASET_SPEC mục G). Không cào trực tiếp Google Images (Google chặn IP Colab + cấm trong điều khoản) — DuckDuckGo cho kết quả tương đương.

In [ ]:
# 0. Drive + code + thư viện
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/snapget-ml'
OUT = f'{WORK}/data/vn_web'          # thư mục VN của ảnh cào (chỉ có train/) — dùng ở notebook train: --vn {OUT}
import os; os.makedirs(OUT, exist_ok=True)
%cd /content
!rm -rf datn && git clone --depth 1 https://github.com/HuyenVt001/datn.git datn
%cd /content/datn/ml
!pip install -q ultralytics ddgs requests pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Giải nén tập VN có sẵn (`vn.rar` / `vn.zip` trên Drive)
Đặt file vào `WORK/data/` (Drive). Ô này giải nén ra `/content/ext_vn` và tìm thư mục VN (có `train/` hoặc `test/`).

In [ ]:
import glob, os
VN_SRC = None
for z in (f'{WORK}/data/vn.rar', f'{WORK}/data/vn.zip'):
    if os.path.exists(z):
        !rm -rf /content/ext_vn && mkdir -p /content/ext_vn
        if z.endswith('.rar'):
            !which unrar >/dev/null || apt-get -qq install -y unrar >/dev/null
            !unrar x -o+ -inul "{z}" /content/ext_vn/
        else:
            !unzip -q -o "{z}" -d /content/ext_vn
        break
cands = [p for p in glob.glob('/content/ext_vn/**/', recursive=True)
         if os.path.isdir(os.path.join(p, 'train', 'images')) or os.path.isdir(os.path.join(p, 'test', 'images'))]
VN_SRC = sorted(cands, key=len)[0].rstrip('/') if cands else None
print('Thư mục VN:', VN_SRC or '(không có vn.rar/vn.zip — bỏ qua mục 1–1c)')

## 1b. LÀM SẠCH dữ liệu (`18_clean_dataset.py`) — chạy TRƯỚC mọi bước khác
Kiểm tra ảnh hỏng, nhỏ hơn 1024px, ảnh banner/ghép, nền trắng studio, ảnh mờ, **ảnh trùng** (y hệt + gần trùng), **ảnh train trùng ảnh test** (rò rỉ), và **ảnh TEST không có dấu vết chụp bằng điện thoại** (không EXIF máy ảnh / có url nguồn ⇒ nghi là ảnh Internet — DATASET_SPEC: test phải 100% tự chụp).
- Lần 1 (ô dưới, **dry-run**): chỉ báo cáo, không đụng file. Đọc bảng `clean_report.csv`.
- Lần 2: đổi `APPLY = True` → ảnh lỗi chuyển vào `VN_SRC/_quarantine/<lý do>/` (**không xoá**), sửa `metadata.csv` (bản cũ `.bak`), tính lại `lighting/distance` từ ảnh thay cho giá trị ngẫu nhiên.

In [ ]:
APPLY = False                    # True sau khi đã xem báo cáo dry-run
QUARANTINE_TEST_WEB = True       # chuyển ảnh test nghi là ảnh Internet ra khỏi test
if VN_SRC:
    flags = '--apply --recompute-metadata' if APPLY else ''
    flags += ' --quarantine-test-web' if QUARANTINE_TEST_WEB else ''
    !python scripts/18_clean_dataset.py --vn "{VN_SRC}" {flags}
    import pandas as pd
    rep = pd.read_csv(f'{VN_SRC}/clean_report.csv')
    print(rep.groupby(['split', 'action']).size())
    display(rep.head(30))

## 1c. Gán nhãn ảnh đã làm sạch bằng 2 model (`17 --import-dir`)
Ảnh train trong `vn.rar` (và ảnh "test" nghi là ảnh Internet ở `_quarantine/test_web`) được đưa qua **cùng quy trình** với ảnh cào: lọc, khử trùng với ảnh test thật, gán box YOLO11x + YOLO-World, accept → `OUT/train`, nghi vấn → `OUT/review`. Ảnh tên `self_` tải từ mạng được đổi thành `gg_`. File gốc **không** bị sửa. Chạy lại = bỏ qua ảnh đã import.
> ⚠️ Sau bước này, **tập test chỉ còn ảnh thật sự tự chụp** — thiếu thì phải chụp bù (không thể thay bằng ảnh Internet).

In [ ]:
TEST_ARGS = ''
if VN_SRC:
    tests = [d for d in (f'{VN_SRC}/test/images',) if os.path.isdir(d)]
    TEST_ARGS = ' '.join(f'--test-dir "{d}"' for d in tests)
    imports = [d for d in (f'{VN_SRC}/train/images', f'{VN_SRC}/_quarantine/test_web/test/images') if os.path.isdir(d)]
    imp = ' '.join(f'--import-dir "{d}"' for d in imports)
    meta = f'--import-meta "{VN_SRC}/metadata.csv"' if os.path.exists(f'{VN_SRC}/metadata.csv') else ''
    !python scripts/17_crawl_autolabel.py --out {OUT} {imp} {meta} {TEST_ARGS} --device 0
print('test dirs (chống rò rỉ cho ô 3):', TEST_ARGS or '(chưa có)')

## 1d. Lưu tập TEST đã làm sạch lên Drive → `WORK/data/vn_clean.zip`
`/content` mất khi Colab ngắt — phải lưu lại. Chỉ gồm `test/` + `metadata.csv` + file tên lớp (ảnh train đã được gán nhãn và nằm trong `OUT` = `vn_web`). Notebook train **ưu tiên `vn_clean.zip`** hơn `vn.zip`/`vn.rar`. Chụp bù ảnh test thật thì thêm vào rồi chạy lại ô này.

In [ ]:
if VN_SRC and APPLY:
    extra = ' '.join(f for f in ('metadata.csv', 'classes.txt', 'data.yaml') if os.path.exists(f'{VN_SRC}/{f}'))
    !cd "{VN_SRC}" && rm -f /content/vn_clean.zip && zip -q -r /content/vn_clean.zip test {extra}
    !cp /content/vn_clean.zip {WORK}/data/vn_clean.zip && ls -la {WORK}/data/vn_clean.zip
    n = len(glob.glob(f'{VN_SRC}/test/images/*'))
    print(f'Tập test sạch còn {n} ảnh (đặc tả: 50 ảnh/lớp × 12 + 100 negative ≈ 700).')
else:
    print('Bỏ qua: chạy ô 1b với APPLY = True trước.')

## 2. API key các nguồn ảnh (miễn phí) — đặt trong Colab *Secrets* (biểu tượng 🔑 bên trái), **không commit, không dán vào chat**
| Secret | Nguồn | Lấy ở đâu |
|---|---|---|
| `SERPAPI_KEY` | **Google Images** (qua SerpApi — cách hợp pháp duy nhất còn lại; Google Custom Search API đã đóng với tài khoản mới) | serpapi.com → Sign up → *API key* (gói free giới hạn số lượt/tháng, mỗi lượt ~100 ảnh) |
| `FLICKR_API_KEY` | Flickr (ảnh CC thật, nhiều ảnh đời thường VN) | flickr.com/services/apps/create → *Apply for a non-commercial key* |
| `PEXELS_API_KEY` | Pexels | pexels.com/api → *Your API Key* |
| `PIXABAY_API_KEY` | Pixabay | pixabay.com/api/docs (đăng nhập → key hiện trong trang docs) |
| `OPENVERSE_CLIENT_ID` / `_SECRET` | Openverse (tuỳ chọn, giảm 504/429) | docs.openverse.org → register |

Không có key nào thì vẫn chạy được với DuckDuckGo + Wikimedia + Openverse. Nhớ bật *Notebook access* cho từng secret.

In [ ]:
import os
KEYS = ('SERPAPI_KEY', 'FLICKR_API_KEY', 'PEXELS_API_KEY', 'PIXABAY_API_KEY', 'OPENVERSE_CLIENT_ID', 'OPENVERSE_CLIENT_SECRET')
try:
    from google.colab import userdata
    for k in KEYS:
        try:
            v = userdata.get(k)
        except Exception:
            v = None
        if v:
            os.environ[k] = v
except Exception:
    pass
for k in KEYS:
    print(f'{k:24s}', 'có' if os.environ.get(k) else '—')

## 3. Cào + gán nhãn — chạy lại ô này để chạy TIẾP sau khi Colab ngắt
- `--per-class 100`: số ảnh **accept** cần có mỗi lớp (đặc tả: VN-train ≥ 150/lớp, trong đó ≥ 50 tự chụp ⇒ ảnh cào ~100).
- `--negatives 150 --hard-negatives 150` (đặc tả: 300 negative, ≥ 150 hard).
- Thử nhanh 1 lớp trước: `--classes chair --per-class 10`.
- `--sources auto` = mọi nguồn có key ở ô 2 (Google/SerpApi, Flickr, Pexels, Pixabay) + DuckDuckGo + Wikimedia + Openverse (cuối cùng). Key sai / hết quota → tự tắt nguồn đó, chạy tiếp bằng nguồn khác.
- Openverse hay báo `504` (server của họ quá tải): script tự thử lại, lỗi 3 lần liên tiếp thì tạm tắt 15 phút. Muốn bỏ hẳn: `--sources serpapi,flickr,ddg,wikimedia`.
- Flickr chỉ lấy ảnh có toạ độ trong VN: thêm `--flickr-vn-only` (ít ảnh hơn nhưng chắc bối cảnh VN).
- Ảnh đã có (từ ô 1c hoặc lần chạy trước) được tính vào số lượng — chỉ cào phần còn thiếu, **không ghi đè** ảnh cũ.
- Thiếu ảnh 1 lớp → thêm từ khoá trong `ml/crawl_queries.yaml` (fork/sửa repo) hoặc tăng `--per-query`.

In [ ]:
# Colab de NGUYEN chu '{TEN_BIEN}' neu bien chua co (vd bo qua o 1c trong phien nay) -> luon dinh nghia truoc
TEST_ARGS = globals().get('TEST_ARGS', '')   # chi co khi da chay o 1c trong phien nay (khu trung voi anh test)
CLASSES_ARG = ''          # vd '--classes backpack' de chi cao bu 1 lop
!python scripts/17_crawl_autolabel.py --out {OUT} --per-class 100 --negatives 150 --hard-negatives 150 \
    --sources auto --per-query 40 --device 0 {TEST_ARGS} {CLASSES_ARG}

## 4. Xem mẫu ~10% ảnh accept (kiểm tra chất lượng nhãn)
Box đỏ = nhãn tự động. Thấy sai nhiều ở lớp nào → báo Claude / tăng `--accept` (chặt hơn) hoặc đưa lớp đó qua review.

In [ ]:
import random, sys
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
sys.path.insert(0, '/content/datn/ml')
from snapget12 import CLASSES
from snapget12.yolo import read_yolo_labels, find_names
names = find_names(OUT) or CLASSES
imgs = sorted(Path(f'{OUT}/train/images').glob('gg_*.jpg'))
sample = random.sample(imgs, min(len(imgs), max(12, len(imgs) // 10)))
for i in range(0, len(sample), 4):
    tiles = []
    for p in sample[i:i + 4]:
        im = Image.open(p).convert('RGB'); im.thumbnail((400, 400)); d = ImageDraw.Draw(im); W, H = im.size
        for c, x, y, w, h in read_yolo_labels(Path(f'{OUT}/train/labels') / (p.stem + '.txt')):
            d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=(255, 0, 0), width=3)
            d.text(((x - w / 2) * W + 3, (y - h / 2) * H + 3), names[c], fill=(255, 255, 0))
        d.text((4, H - 14), p.name, fill=(0, 255, 0)); tiles.append(im)
    grid = Image.new('RGB', (1600, 400), (30, 30, 30))
    for k, t in enumerate(tiles):
        grid.paste(t, (k * 400, 0))
    display(grid)

## 4b. TỰ DUYỆT ảnh review (thay cho duyệt tay — ~20–40 phút trên T4)
Với mỗi ảnh trong `review/` của **lớp còn thiếu**: cắt vùng quanh box *chưa chắc* (điểm thấp, chỉ 1 model thấy, box lớp khác nghi vấn), phóng to, cho **2 model nhìn lại**:
- box được xác nhận trên vùng phóng to → giữ; không xác nhận → xoá;
- laptop: tìm bàn phím trong vùng laptop → thêm box `keyboard`; không thấy → coi là laptop gập/góc nghiêng;
- ảnh vẫn còn nghi vấn, hoặc dính **hard negative / màn hình** → để nguyên trong `review/` (không dùng).

Ảnh được chuyển sang `train/` với `autolabel = auto-review` trong `metadata.csv` (lọc/hoàn tác được). Dừng khi lớp đủ `--per-class`. Muốn chạy cả lớp đã đủ: thêm `--auto-review-all`.
> Ghi vào báo cáo: phần ảnh này là nhãn tự động 2 tầng (không người duyệt) — sai số chấp nhận cho tập train bổ sung; tập TEST vẫn phải người gán.

In [ ]:
!python scripts/17_crawl_autolabel.py --out {OUT} --auto-review --per-class 100 --device 0

In [ ]:
# Xem mẫu ảnh TỰ DUYỆT — lướt ~2 phút: thấy sai nhiều ở lớp nào thì báo Claude
import csv, random
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
from snapget12 import CLASSES
from snapget12.yolo import read_yolo_labels, find_names
names = find_names(OUT) or CLASSES
with open(f'{OUT}/metadata.csv', encoding='utf-8') as f:
    auto = [r['filename'] for r in csv.DictReader(f) if r.get('autolabel') == 'auto-review']
print(len(auto), 'anh tu duyet')
sample = random.sample(auto, min(len(auto), 32))
for i in range(0, len(sample), 4):
    grid = Image.new('RGB', (1600, 400), (30, 30, 30))
    for k, fn in enumerate(sample[i:i + 4]):
        p = Path(f'{OUT}/train/images/{fn}')
        if not p.exists():
            continue
        im = Image.open(p).convert('RGB'); im.thumbnail((400, 400)); d = ImageDraw.Draw(im); W, H = im.size
        for c, x, y, w, h in read_yolo_labels(Path(f'{OUT}/train/labels') / (p.stem + '.txt')):
            d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=(0, 200, 255), width=3)
            d.text(((x - w / 2) * W + 3, (y - h / 2) * H + 3), names[c], fill=(255, 255, 0))
        d.text((4, H - 14), fn, fill=(0, 255, 0)); grid.paste(im, (k * 400, 0))
    display(grid)

## 5. (Tuỳ chọn — chỉ khi 4b vẫn chưa đủ) Ảnh REVIEW → sửa tay trên Roboflow
1. Ô dưới **chỉ zip ảnh review của các lớp còn THIẾU** so với `PER_CLASS` (mặc định lấy gấp ~1.5 lần số còn thiếu, vì sẽ phải xoá bớt ảnh xấu) → `WORK/data/vn_web_review_pick.zip`. Không cần duyệt hết vài nghìn ảnh review. Muốn zip toàn bộ: `REVIEW_ALL = True`.
2. Roboflow → *Create Project* (Object Detection) → *Upload* zip → box nháp hiện sẵn. Xem lý do từng ảnh trong `review/review_queue.csv` (vd "box điểm thấp", "umbrella trùng 'conical hat'", "laptop chưa có box keyboard"). Sửa theo DATASET_SPEC mục B, D; **xoá** ảnh không dùng được.
3. *Generate version*: **không** Preprocessing/Augmentation, 100% train → *Export* YOLOv8 → tải zip về, kéo lên Drive `WORK/data/review_fixed.zip`.

In [ ]:
import csv, math, os, random, shutil
from collections import Counter, defaultdict
from pathlib import Path
PER_CLASS = {'neg': 150, 'hardneg': 150}   # lop khac: 100 (giong --per-class o 3)
PICK_FACTOR = 1.5
REVIEW_ALL = False
if REVIEW_ALL:
    !cd {OUT} && zip -q -r {WORK}/data/vn_web_review.zip review -x 'review/metadata_pending.csv'
    !ls -la {WORK}/data/vn_web_review.zip
else:
    acc = Counter()
    with open(f'{OUT}/metadata.csv', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            acc[r['filename'].split('_', 1)[1].rsplit('_', 1)[0]] += 1     # gg_potted_plant_0012.jpg -> potted_plant
    queue = defaultdict(list)
    with open(f'{OUT}/review/review_queue.csv', encoding='utf-8') as f:
        for r in csv.DictReader(f):
            if os.path.exists(f'{OUT}/review/images/{r["filename"]}'):
                queue[r['group'].replace(' ', '_')].append(r)
    pick_dir = Path('/content/review_pick'); shutil.rmtree(pick_dir, ignore_errors=True)
    (pick_dir / 'images').mkdir(parents=True); (pick_dir / 'labels').mkdir()
    random.seed(0); picked = []
    for g, rows in sorted(queue.items()):
        need = PER_CLASS.get(g, 100) - acc[g]
        if need <= 0:
            continue
        take = random.sample(rows, min(len(rows), math.ceil(need * PICK_FACTOR)))
        print(f'{g:14s} dang co {acc[g]:3d} accept, thieu {need:3d} -> lay {len(take):3d}/{len(rows)} anh review')
        for r in take:
            shutil.copy(f'{OUT}/review/images/{r["filename"]}', pick_dir / 'images')
            lab = Path(f'{OUT}/review/labels') / (Path(r['filename']).stem + '.txt')
            if lab.exists():
                shutil.copy(lab, pick_dir / 'labels')
            picked.append(r)
    for f in ('classes.txt', 'data.yaml'):
        if os.path.exists(f'{OUT}/review/{f}'):
            shutil.copy(f'{OUT}/review/{f}', pick_dir)
    with open(pick_dir / 'review_queue.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['filename', 'group', 'reasons', 'url']); w.writeheader(); w.writerows(picked)
    print(f'Tong {len(picked)} anh can duyet')
    !cd /content && rm -f review_pick.zip && zip -q -r review_pick.zip review_pick && cp review_pick.zip {WORK}/data/vn_web_review_pick.zip
    !ls -la {WORK}/data/vn_web_review_pick.zip

## 6. Đưa ảnh đã sửa vào train (`--promote`)
Dùng ảnh gốc độ phân giải cao trong `review/` + nhãn từ bản export (tên file `.rf.xxx` của Roboflow được xử lý tự động).

In [ ]:
!rm -rf /content/review_fixed && unzip -q -o {WORK}/data/review_fixed.zip -d /content/review_fixed
!python scripts/17_crawl_autolabel.py --out {OUT} --promote /content/review_fixed

## 7. Báo cáo
`crawl_report.json` (số ảnh, lý do lọc/loại, độ đa dạng tối/xa/bối cảnh từng lớp), `data_sources.md` + `web_attributions.csv` (trích nguồn cho báo cáo). Dùng trong notebook train: ô 4 thêm `--vn {OUT}` (ảnh cào) bên cạnh `--vn` ảnh tự chụp.

In [ ]:
import json
r = json.load(open(f'{OUT}/crawl_report.json'))
print('accept:', r['accepted']); print('review:', r['review'])
for g, v in r['runs'].items():
    print(g, '| loai:', dict(sorted(v['reject'].items(), key=lambda x: -x[1])[:3]), '| loc:', dict(sorted(v['filtered'].items(), key=lambda x: -x[1])[:4]))
print(open(f'{OUT}/data_sources.md').read())

## 8. ĐÓNG GÓI → `WORK/data/snapget_vn_supplement.zip` (sản phẩm cuối của notebook này)
Gom mọi ảnh **đã dùng được** (`train/`: accept + tự duyệt + đã sửa) thành 1 dataset YOLO gọn: `train/images`, `train/labels`, `data.yaml` (12 lớp), `metadata.csv`, `web_attributions.csv` (trích nguồn), `README.md` (thẻ dataset: số lượng, cách thu thập/gán nhãn, giấy phép), `stats.json`. Bỏ `review/`, ảnh bị loại, file trạng thái cào.
Notebook train **tự dùng file zip này** (ưu tiên hơn thư mục `vn_web`). Cào/duyệt thêm → chạy lại ô này để cập nhật.

In [ ]:
!python scripts/19_package_dataset.py --src {OUT} --out {WORK}/data/snapget_vn_supplement.zip
!unzip -p {WORK}/data/snapget_vn_supplement.zip snapget_vn_supplement/README.md | head -60